# Módulo A — Recorrido del flujo de pronóstico del gasto

Este notebook recorre, **fase por fase y en orden**, el Módulo A del *SistemaPrediccionOC*: el pronóstico del gasto público mensual en las Órdenes de Compra de los Acuerdos Marco de PERÚ COMPRAS, **desde la extracción hasta el entrenamiento y el pronóstico**.

| # | Fase | Módulo |
|---|------|--------|
| 1 | Ingesta / ETL | `src.fases.f01_ingesta` |
| 2 | Limpieza y validación | `src.fases.f02_limpieza` |
| 3 | Serie temporal | `src.fases.f03_serie_temporal` |
| 4 | Análisis exploratorio (EDA) | `src.fases.f04_eda` |
| 5 | Features de calendario | `src.fases.f05_features` |
| 6 | Modelado | `src.fases.f06_modelado` |
| 7 | Evaluación y pronóstico | `src.fases.f07_evaluacion` |

> Para regenerar **todos** los artefactos de una vez basta con `python main.py` (o `from src.pipeline import ejecutar_pipeline; ejecutar_pipeline()`). Aquí ejecutamos las fases una a una para *ver* cada paso.

In [ ]:
# Configuración: permitir importar el paquete src/ desde notebooks/
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src import config, figuras
from src.fases import f01_ingesta as ingesta
from src.fases import f02_limpieza as limpieza
from src.fases import f03_serie_temporal as serie_temporal
from src.fases import f04_eda as eda
from src.fases import f05_features as features
from src.fases import f06_modelado as modelado
from src.fases import f07_evaluacion as evaluacion

from IPython.display import Image, Markdown, display

figuras.aplicar_estilo()
modelado.fijar_semillas()

def mostrar(slug):
    """Muestra inline la figura `slug` del registro central (numerada y ordenada)."""
    display(Image(filename=str(config.DIR_FIGURAS / figuras.nombre_archivo(slug))))

print('Entorno listo. Figuras del catálogo:', len(figuras.CATALOGO))

## Fase 1 — Ingesta / ETL

Lee de forma recursiva todos los CSV mensuales de `data/`, detecta la codificación por archivo, normaliza columnas y los consolida en un único DataFrame.

In [ ]:
df_crudo, rep_ingesta = ingesta.consolidar(guardar=True)
print(f"Archivos leídos : {rep_ingesta['archivos_leidos']}/{rep_ingesta['archivos_encontrados']}")
print(f"Filas           : {rep_ingesta['filas_totales']:,}")
print(f"Meses           : {rep_ingesta['n_meses']} ({rep_ingesta['meses_cubiertos'][0]} → {rep_ingesta['meses_cubiertos'][-1]})")
print(f"Codificaciones  : {rep_ingesta['codificaciones']}")
df_crudo.head(3)

## Fase 2 — Limpieza y validación

Tipa fechas y montos, quita duplicados, descarta órdenes sin fecha/monto y excluye los estados que no representan gasto efectivo (`RESUELTA`, `ORDEN DE COMPRA NULA`).

In [ ]:
df_limpio, rep_limpieza = limpieza.limpiar(df_crudo, guardar=True)
print(f"Filas iniciales : {rep_limpieza['filas_iniciales']:,}")
print(f"Excluidas estado: {rep_limpieza.get('ordenes_excluidas_por_estado', 0):,}")
print(f"Filas válidas   : {rep_limpieza['filas_finales']:,}")
df_limpio[[config.COL_FECHA_FORMALIZACION, config.COL_TOTAL, config.COL_ESTADO]].head(3)

## Fase 3 — Serie temporal

Construye la **serie mensual del gasto total** (variable objetivo), detecta meses incompletos en la cola, y arma el **panel por categoría** y los **drivers internos** que alimentan el modelo global.

In [ ]:
serie_completa = serie_temporal.construir_serie_total(df_limpio)
deteccion = serie_temporal.detectar_meses_incompletos(serie_completa)
serie_total, serie_categoria, _ = serie_temporal.construir_series(df_limpio, guardar=True)

ultimo_mes = serie_total.index[-1]
panel = serie_temporal.construir_panel_categorias(df_limpio)
panel = panel[panel['fecha'] <= ultimo_mes].copy()
drivers = serie_temporal.construir_drivers_mensuales(df_limpio)
drivers = drivers[drivers.index <= ultimo_mes].copy()

print('Meses:', len(serie_total), '| incompletos:', deteccion['meses_incompletos'] or 'ninguno')
print('Categorías en panel:', panel['categoria'].nunique())
serie_total.tail(6)

## Fase 4 — Análisis exploratorio (EDA)

Genera las figuras **01–08** (numeradas por el registro central `figuras`) con sus interpretaciones, que se incorporan a la sección 2 del informe.

In [ ]:
md_eda = eda.ejecutar_eda(df_limpio, serie_completa, deteccion, rep_ingesta, rep_limpieza)
for slug in figuras.CATALOGO[:8]:
    mostrar(slug)

## Fase 5 — Features de calendario

Materializa los predictores de **calendario** (días hábiles, indicadores de mes, armónicos de Fourier). Son el insumo con que los modelos de árboles *aprenden* el desplome de enero en lugar de promediarlo.

In [ ]:
tabla_cal = features.tabla_calendario(serie_total.index, 'mes')
print('Columnas:', list(tabla_cal.columns))
tabla_cal.tail(6)

## Fase 6 — Modelado

Catálogo de modelos de pronóstico a entrenar y comparar (líneas base, estadísticos, árboles de gradiente y redes). El entrenamiento real ocurre dentro del backtest de la fase 7.

In [ ]:
for i, nombre in enumerate(modelado.MODELOS, 1):
    print(f'{i:2d}. {nombre}')

## Fase 7 — Evaluación y pronóstico

Backtesting de **origen móvil**, selección del mejor modelo por **WAPE/MASE**, pronóstico final con intervalo, y figuras **09–11**. Genera las secciones 3–4 del informe.

> Esta celda es la más pesada (incluye Optuna y el backtest de todos los modelos).

In [ ]:
md_modelos, resultados = evaluacion.ejecutar_evaluacion(
    serie_total, df_limpio, panel=panel, drivers=drivers
)
print('Mejor modelo :', resultados['mejor_modelo'])
print('Intervalos   :', resultados.get('metodo_intervalo'))
resultados['tabla_metricas'].round(2)

In [ ]:
# Figuras de evaluación y pronóstico (09–11)
for slug in ['holdout_real_vs_modelos', 'backtest_error_por_mes', 'pronostico_final']:
    mostrar(slug)

In [ ]:
# Pronóstico final con intervalo de confianza
resultados['pronostico']

---

El informe completo (todas las fases, figuras e interpretaciones) queda en [`outputs/resultados/informe_resultados.md`](../outputs/resultados/informe_resultados.md).